# B · Low Carbon London: what is the data?

Part B tests the BL01 baseline on real households. Nobody publishes meter data for DFS
participants, so we use **Low Carbon London (LCL)**: UK Power Networks metered about 5,500
London homes every half-hour from late 2011 to early 2014. Through 2013 about 1,100 of them
were on a **dynamic time-of-use tariff (ToU)**: a day-ahead price signal set each half-hour
to **High** (67.2p/kWh), **Low** (4.0p) or **Normal** (11.8p). The rest stayed on a flat
tariff (**Std**) and act as a control group.

That gives Part B two things DFS data can't:
- homes where the true saving is known to be zero (Std homes never saw a price), and
- a real price response measured against a control group (ToU vs Std).

The catch: this is 2013 and a different scheme, so it tests the *baseline method*, not
DFS participants.

This notebook only looks at the data: the raw files, one home, the price events, coverage,
what an average home looks like, and what Part B has to work with. No BL01 yet.

**Raw data is a manual download** (`data/` is not committed): get `LCL-FullData.zip` and
`Tariffs.xlsx` from the
[London Datastore](https://data.london.gov.uk/dataset/smartmeter-energy-consumption-data-in-london-households-vqm0d)
into `data/raw/lcl/`. The first code cell converts them once with `uk_dfs.data.lcl`.

In [ ]:
import subprocess
from io import StringIO

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Project colours; importing _common also applies the chart style
from _common import AQUA, BLUE, GRID, INK2, ORANGE, YELLOW, load_london_weather

from uk_dfs.data import lcl

if not (lcl.RAW / "LCL-FullData.zip").exists():
    raise FileNotFoundError(f"Download LCL-FullData.zip and Tariffs.xlsx into {lcl.RAW}")

# Convert the raw CSV once (first run: roughly half an hour; after that: instant)
lcl.build()  # readings.parquet (every reading), households.parquet (one row per home)
lcl.build_aggregates()  # per-slot group totals, per-home daily totals
lcl.build_cube()  # home × day × half-hour array, used by the BL01 work

TZ = "Europe/London"
TRIAL = (pd.Timestamp("2013-01-01"), pd.Timestamp("2014-01-01"))

## The raw file: one 8.5 GB CSV
One row per home per half-hour, all ~5,500 homes in one file (167 million rows), sorted by
home and then time. Read the first 10 rows straight out of the zip:

In [ ]:
# The archive uses Deflate64, which Python's zipfile can't read, so stream it with `unzip`
cmd = ["unzip", "-p", str(lcl.RAW / "LCL-FullData.zip")]
with subprocess.Popen(cmd, stdout=subprocess.PIPE, text=True) as p:
    lines = [p.stdout.readline() for _ in range(11)]
    p.kill()  # don't decompress the other 8.5 GB
raw = pd.read_csv(StringIO("".join(lines)), dtype=str)
raw

Things to know about the raw columns:
- `LCLid`: the home. `stdorToU`: **Std** (flat tariff) or **ToU** (dynamic tariff in 2013).
- `DateTime`: **GMT all year** (no summer time), stamped at the **end** of the half-hour,
  with a 7-digit fraction of a second.
- `KWH/hh (per half hour) `: energy in that half-hour, with a trailing space in the
  header; missing readings are the string `Null`.

`lcl.build()` turns this into compact Parquet: `household` (MAC002346 → 2346), `ts` and
`kwh`, with `ts` moved back 30 minutes to the **start** of the half-hour (see below). A
few dozen readings off the half-hour grid and duplicate rows are dropped later.

## Households
One row per home, with its tariff group.

In [ ]:
homes = pd.read_parquet(lcl.HOUSEHOLDS)
print(homes.tariff.value_counts().to_string())
homes.head()

## One home, every reading
The readings file holds 168 million rows; read one home with a filter. We pick a **typical
ToU home**: a full 2013 (at least 360 complete days) and median daily use among those.

In [ ]:
daily = pd.read_parquet(lcl.DAILY).merge(homes, on="household")  # kWh per home per day
full13 = daily[(daily["count"] == 48) & (daily.date.dt.year == 2013)]
per_home = full13.groupby(["household", "tariff"])["sum"].agg(["size", "mean"]).reset_index()
cands = per_home[(per_home.tariff == "ToU") & (per_home["size"] >= 360)]
HOME = int(cands.household.iloc[(cands["mean"] - cands["mean"].median()).abs().argmin()])

one = pd.read_parquet(lcl.READINGS, filters=[("household", "==", HOME)])
one["start"] = one.ts.dt.tz_localize("UTC").dt.tz_convert(TZ).dt.tz_localize(None)
print(f"Home {HOME}: {len(one):,} readings, {one.ts.min():%d %b %Y} – {one.ts.max():%d %b %Y}, "
      f"{per_home.set_index('household').loc[HOME, 'mean']:.1f} kWh a day in 2013")  # fmt: skip
one.head()

## The price signal (`Tariffs.xlsx`)
One row per half-hour of 2013: **Normal**, **High** or **Low**. The time column is GMT, like
the readings. Consecutive half-hours with the same signal form one **price event**; High
plays the part of a DFS turn-down event, Low a turn-up event.

In [ ]:
tariffs = lcl.load_tariffs()
tariffs["start"] = tariffs.ts.dt.tz_localize("UTC").dt.tz_convert(TZ).dt.tz_localize(None)
print(tariffs.signal.value_counts().to_string())

# One row per event. Group on GMT time, so a clock change doesn't split an event.
s = tariffs[tariffs.signal != "Normal"]
new = (s.ts.diff() != pd.Timedelta("30min")) | (s.signal != s.signal.shift())
events = s.groupby(new.cumsum()).agg(
    signal=("signal", "first"), start=("start", "first"), ts=("ts", "first"),
    halfhours=("ts", "size"),
)  # fmt: skip
events["hours"] = events.pop("halfhours") / 2
events["end"] = events.start + pd.to_timedelta(events.hours, unit="h")
events = events.reset_index(drop=True)
events.groupby("signal").hours.describe()

### Plot 1: one home, one week
Home picked above, Mon 18 – Sun 24 Feb 2013, every half-hour. Orange bands are High
half-hours (early Monday, then Wed and Thu evening, 17:00–23:00), green bands Low. Each
reading is drawn over the half-hour it covers.

In [ ]:
W0, W1 = pd.Timestamp("2013-02-18"), pd.Timestamp("2013-02-25")
week = one[(one.start >= W0) & (one.start < W1)]

fig, ax = plt.subplots(figsize=(12, 3.8))
for e in events[(events.end > W0) & (events.start < W1)].itertuples():
    colour = ORANGE if e.signal == "High" else AQUA
    ax.axvspan(e.start, e.end, color=colour, alpha=0.18, lw=0)
ax.plot(week.start, week.kwh * 2, color=BLUE, lw=1.2, drawstyle="steps-post")
ax.plot([], [], color=ORANGE, alpha=0.4, lw=8, label="High price")
ax.plot([], [], color=AQUA, alpha=0.4, lw=8, label="Low price")
ax.set_ylabel("Average kW in the half-hour")
ax.set_title(f"Home {HOME} (ToU): one week, every half-hour", fontsize=10)
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
ax.legend(loc="upper left", fontsize=8);

### Plot 2: when the price events ran
**Left:** every 2013 event as a vertical bar from start to end time (events past midnight
are cut at 24:00). The grey band is when DFS turn-down events usually run (16:00–20:00).
**Right:** how long events lasted. DFS events last 1–2 hours.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), width_ratios=[2.3, 1])
ax1.axhspan(16, 20, color=GRID, alpha=0.7, lw=0, label="Usual DFS hours")
for k, colour in (("High", ORANGE), ("Low", AQUA)):
    e = events[events.signal == k]
    h0 = e.start.dt.hour + e.start.dt.minute / 60
    ax1.vlines(e.start.dt.normalize(), h0, np.minimum(h0 + e.hours, 24), color=colour, lw=2.2,
               label=f"{k} ({len(e)} events)")  # fmt: skip
ax1.set_ylim(0, 24)
ax1.set_yticks(range(0, 25, 4), [f"{h:02d}:00" for h in range(0, 25, 4)])
ax1.set_ylabel("Time of day (local)")
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax1.set_title("2013 price events: around the clock, all year", fontsize=10)
ax1.legend(loc="upper left", bbox_to_anchor=(0, -0.08), ncol=3, fontsize=8)

bins = np.arange(0, 25, 1.5)
for k, colour in (("High", ORANGE), ("Low", AQUA)):
    ax2.hist(events[events.signal == k].hours, bins=bins, color=colour, alpha=0.6, label=k)
ax2.axvspan(1, 2, color=GRID, alpha=0.9, lw=0, label="DFS: 1–2 h")
ax2.set_xlabel("Event length, hours")
ax2.set_ylabel("Events")
ax2.set_title("Events last 3–24 hours", fontsize=10)
ax2.legend(fontsize=8)
fig.tight_layout()

hi = events[events.signal == "High"]
dfs_like = hi[(hi.start.dt.dayofweek < 5) & hi.start.dt.hour.between(16, 19)]
print(f"High events starting on a weekday between 16:00 and 19:59: {len(dfs_like)} of {len(hi)}")
dfs_like[["start", "hours"]]

## Coverage
### Plot 3: how many homes report, day by day
A home counts on a day if it has at least 46 of that day's 48 readings (clock-change days
have 46 or 50 local half-hours, because the readings are in GMT).

In [ ]:
ok = daily[daily["count"] >= 46]
cnt = ok.groupby(["date", "tariff"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axvspan(*TRIAL, color=GRID, alpha=0.5, lw=0, label="dToU trial (2013)")
for k, colour, label in (("Std", BLUE, "Flat tariff (Std)"), ("ToU", YELLOW, "Time-of-use (ToU)")):
    ax.plot(cnt.index, cnt[k], color=colour, label=label)
ax.set_ylabel("Homes with a full day")
ax.set_title("Homes join through 2012; most report through 2013", fontsize=10)
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper left", fontsize=8);

### Data quality
From the home × day × half-hour cube (`lcl.build_cube()`: all ToU homes plus a fixed random
1,500 Std homes). Share of missing readings and of exact zeros, by group, for 2013.

In [ ]:
X, cube_homes, cube_tou, cube_dates = lcl.load_cube()  # X is memory-mapped, kWh
in13 = (cube_dates >= TRIAL[0]) & (cube_dates < TRIAL[1])
quality = {}
for name, rows in (("Std", ~cube_tou), ("ToU", cube_tou)):
    x = np.asarray(X[rows][:, in13])
    present = ~np.isnan(x)
    quality[name] = {
        "homes": f"{rows.sum():,}",
        "missing readings": f"{1 - present.mean():.1%}",
        "zero readings": f"{(x[present] == 0).mean():.1%}",
        "homes with no 2013 data": f"{(~present.any(axis=(1, 2))).mean():.1%}",
    }
pd.DataFrame(quality).T

## What an average home looks like
Group averages per half-hour (`slot_means.parquet`): the mean kWh per home in each group,
for every half-hour of the data, converted to local time.

In [ ]:
sm = pd.read_parquet(lcl.SLOT_MEANS).pivot(index="ts", columns="tou", values=["sum", "count"])
avg = pd.DataFrame({
    "std": sm["sum"][False] / sm["count"][False],  # mean kWh per home per half-hour
    "tou": sm["sum"][True] / sm["count"][True],
    "n_std": sm["count"][False],
    "n_tou": sm["count"][True],
})  # fmt: skip
avg["signal"] = tariffs.set_index("ts").signal.reindex(avg.index)
avg.index = avg.index.tz_localize("UTC").tz_convert(TZ).tz_localize(None)
avg = avg[(avg.n_std >= 1000) & (avg.n_tou >= 300)]  # enough homes in both groups
avg["hour"] = avg.index.hour + avg.index.minute / 60
print(f"Enough homes in both groups: {avg.index.min():%d %b %Y} – {avg.index.max():%d %b %Y}")
avg.head()

### Plot 4: the average weekday home, by season
Std group, average power in each half-hour. The grey band is the usual DFS window.

In [ ]:
SEASONS = {"Winter (DJF)": (12, 1, 2), "Spring (MAM)": (3, 4, 5), "Summer (JJA)": (6, 7, 8),
           "Autumn (SON)": (9, 10, 11)}  # fmt: skip
wk = avg[avg.index.dayofweek < 5]

fig, ax = plt.subplots(figsize=(10, 4))
ax.axvspan(16, 20, color=GRID, alpha=0.7, lw=0, label="Usual DFS hours")
for (name, months), colour in zip(SEASONS.items(), (BLUE, ORANGE, AQUA, YELLOW), strict=True):
    prof = wk[wk.index.month.isin(months)].groupby("hour")["std"].mean() * 2  # kWh/hh -> kW
    ax.plot(prof.index + 0.25, prof, color=colour, label=name)
ax.set_xticks(range(0, 25, 3), [f"{h:02d}:00" for h in range(0, 25, 3)])
ax.set_xlim(0, 24)
ax.set_ylabel("Average home, kW")
ax.set_title("The average home peaks around 18:00–20:00, highest in winter", fontsize=10)
ax.legend(loc="upper left", fontsize=8);

### Plot 5: one home is far noisier than the average
BL01 is calculated **per meter**, so what matters is how much one home's use moves from
day to day, not the average.
- **Left:** homes differ about tenfold in size (mean kWh per day).
- **Right:** for each Std home in the cube, its mean use from 17:00 to 18:00 on weekdays
  against the day-to-day standard deviation of that hour. Points near the dashed line vary
  by about as much as they use.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
size = ok[(ok.date >= TRIAL[0]) & (ok.date < TRIAL[1])].groupby(["household", "tariff"])["sum"]
size = size.mean().reset_index()
bins = np.arange(0, 40.5, 1)
for k, colour in (("Std", BLUE), ("ToU", YELLOW)):
    v = size[size.tariff == k]["sum"]
    ax1.hist(v, bins=bins, density=True, histtype="step", color=colour, lw=2,
             label=f"{k}: median {v.median():.1f} kWh")  # fmt: skip
ax1.set_xlabel("Mean kWh per day, 2013")
ax1.set_yticks([])
ax1.set_ylabel("Share of homes")
ax1.set_title("Homes vary about tenfold in size", fontsize=10)
ax1.legend(fontsize=8)

SLOTS = [34, 35]  # 17:00–17:30 and 17:30–18:00, as stored in the cube
weekday = cube_dates.dayofweek < 5
hour = np.asarray(X[~cube_tou][:, weekday][:, :, SLOTS]).sum(axis=2) * 1000  # kWh -> mean W
hour[np.isnan(np.asarray(X[~cube_tou][:, weekday][:, :, SLOTS])).any(axis=2)] = np.nan
enough = np.sum(~np.isnan(hour), axis=1) >= 100
mean_w, sd_w = np.nanmean(hour[enough], axis=1), np.nanstd(hour[enough], axis=1)
ax2.scatter(mean_w, sd_w, s=6, color=BLUE, alpha=0.4, linewidths=0)
lim = np.nanpercentile(mean_w, 99)
ax2.plot([0, lim], [0, lim], color=INK2, ls="--", lw=1, label="SD = mean")
ax2.set_xlim(0, lim)
ax2.set_ylim(0, lim * 1.2)
ax2.set_xlabel("Mean use 17:00–18:00, weekdays (W)")
ax2.set_ylabel("Day-to-day SD (W)")
ax2.set_title(f"{enough.sum():,} Std homes: median SD is {np.median(sd_w / mean_w):.0%} of "
              "the mean", fontsize=10)  # fmt: skip
ax2.legend(loc="upper left", fontsize=8)
fig.tight_layout()

### Which half-hour does a reading belong to?
**Readings are labelled by the end of the half-hour.** The trial's data user guide describes
the timestamp column of the consumption tables as:

> "Time-stamp at the end of each half-hour measurement period in the GMT time-zone."

(Schofield et al., [*Low Carbon London project: Data from the dynamic time-of-use electricity
pricing trial, 2013*](http://doc.ukdataservice.ac.uk/doc/7857/mrdoc/pdf/7857_userguide.pdf),
UK Data Service SN 7857, Imperial College London, 2015.) A refactored copy of this
London Datastore file reads it the same way: "Measurements are in kWh (energy consumption)
for the *preceding* half hour"
([4TU.ResearchData](https://data.4tu.nl/datasets/fbbe775b-48d8-469f-a39b-b64488bfd6fd)).

**`Tariffs.xlsx` is labelled by the start.** It runs from 2013-01-01 00:00 to 2013-12-31 23:30.

So the raw reading stamped 17:30 covers 17:00–17:30, and goes with the tariff row stamped 17:00.
`lcl.build()` subtracts 30 minutes from every raw stamp, so from the readings table onwards
`ts` marks the start of the half-hour, as in the tariff file. Only the raw CSV rows at the
top still show the original stamps.

### Plot 6: household use against temperature
Daily kWh of the average Std home (from the per-home daily totals) against London's daily
mean temperature (Open-Meteo, observed), weekdays, 2012–2013.

In [ ]:
wx = load_london_weather()
std_day = ok[ok.tariff == "Std"].groupby("date")["sum"].mean().rename("kwh").to_frame()
std_day = std_day.join(wx.temp)
std_day = std_day[(std_day.index.dayofweek < 5) & (std_day.index.year.isin([2012, 2013]))]
std_day = std_day[std_day.index >= avg.index.min().normalize()]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(std_day.temp, std_day.kwh, s=10, color=BLUE, alpha=0.6, linewidths=0)
ax.set_xlabel("Daily mean temperature, °C")
ax.set_ylabel("kWh per home per day (Std)")
cold = std_day[std_day.temp < 2].kwh.mean() / std_day[std_day.temp > 16].kwh.mean()
ax.set_title(f"Homes use {cold:.1f}× as much below 2 °C as above 16 °C", fontsize=10);

## What Part B has to work with

In [ ]:
n_full = ok.groupby(["date", "tariff"]).size().unstack(fill_value=0)
good_days = n_full[(n_full.Std >= 1000) & (n_full.ToU >= 300)].index
n_homes = homes.tariff.value_counts()
scope = pd.Series({
    "Homes in LCL (Std / ToU)": f"{n_homes.Std:,} / {n_homes.ToU:,}",
    "Homes in the cube (Std sample / ToU)": f"{(~cube_tou).sum():,} / {cube_tou.sum():,}",
    "Days with ≥1,000 Std and ≥300 ToU full homes": f"{len(good_days)} "
        f"({good_days.min():%d %b %Y} – {good_days.max():%d %b %Y})",
    "High events, 2013": len(hi),
    "  … starting on a weekday, 16:00–19:59": len(dfs_like),
    "  … of those, 1–2 hours long": int((dfs_like.hours <= 2).sum()),
    "Low events, 2013": int((events.signal == "Low").sum()),
})  # fmt: skip
scope.to_frame("value")

### Takeaways
- **Plenty of homes, clean enough.** 5,566 homes (4,443 Std, 1,123 ToU). Both groups have
  enough homes from Apr 2012 to Feb 2014 (680 days). About 4% of 2013 readings are
  missing, with a few days missing for everyone (the dips in Plot 3).
- **The price events don't look like DFS.** 69 High events lasting 3–24 hours (median 6),
  at all times of day. Only 18 start on a weekday evening (16:00–19:59), mostly 17:00–23:00,
  and none lasts 1–2 hours. Part B's known-effect test therefore uses long events, and a
  DFS-length window sits inside a longer event.
- **One home is very noisy.** Its 17:00–18:00 use moves from day to day by about 72% of
  its average (median home), and homes differ about tenfold in size. A per-meter baseline
  is a rough guess for any single home; the question for Part B is whether the errors
  cancel across a portfolio.
- **Weather matters.** The average home uses 1.6× as much below 2 °C as above 16 °C, so a
  baseline built from milder days will miss on a cold one.
- **Raw readings are end-labelled, the tariff file start-labelled** (documented above).
  `lcl.build()` moves readings back 30 minutes, so everything here lines up with the price
  signal and with DFS settlement periods.